In [5]:
# ==============================================================================
# Cell: Rock-Solid QLoRA Fine-Tuning (Native TRL, Zero-NaN, Full 8-Key Schema)
# ==============================================================================
!pip install -q trl peft bitsandbytes accelerate datasets openpyxl pandas

import os
import json
import re
import shutil
import torch
import pandas as pd
from pathlib import Path
from datasets import load_dataset
from transformers import (
    AutoModelForCausalLM,
    AutoTokenizer,
    BitsAndBytesConfig,
    TrainingArguments,
)
from peft import LoraConfig, get_peft_model, prepare_model_for_kbit_training
from trl import SFTTrainer

# 1. Dataset Discovery
DATASET_PATH = "/kaggle/input/datasets/rakeshadak008/dataset7/CPSE_SIH26099_dataset_backup_original.xlsx"
if not os.path.exists(DATASET_PATH):
    for root, _, files in os.walk("/kaggle/input"):
        for f in files:
            if f.endswith(".xlsx") or f.endswith(".csv"):
                DATASET_PATH = os.path.join(root, f)
                break

print(f"Loading dataset from: {DATASET_PATH}")
df = pd.read_excel(DATASET_PATH)

# 2. Explicit 8-Key Schema System Prompt
SCHEMA_SYSTEM_PROMPT = (
    "You are an expert industrial material master data analyst for Indian Central Public Sector Enterprises (CPSEs). "
    "Extract standardized inventory attributes strictly as valid JSON with these exact 8 keys:\n"
    "{\n"
    '  "Company": "CPSE organization name",\n'
    '  "Item Description (Raw)": "core engineering description",\n'
    '  "Item Code / Legacy Ref": "material code or null",\n'
    '  "Quantity": integer quantity number or null,\n'
    '  "UOM": "unit of measure like NOS, LTR, KGS, SET or null",\n'
    '  "Part Number / OEM Number": "part number or null",\n'
    '  "Make / Brand": "manufacturer brand or null",\n'
    '  "Specifications / Dimensions": "specs and dimensions or null"\n'
    "}"
)

def clean_val(v):
    if pd.isna(v): return None
    s = str(v).strip()
    return None if s.lower() in ["nan", "none", "null", "unknown", "n/a", ""] else s

def row_to_dict(r):
    qty = clean_val(r.get("Quantity"))
    try:
        qty_int = int(float(qty)) if qty is not None else None
    except Exception:
        qty_int = None
    return {
        "Company": clean_val(r.get("Company")),
        "Item Description (Raw)": str(r.get("Item Description (Raw)", "")).strip(),
        "Item Code / Legacy Ref": clean_val(r.get("Item Code / Legacy Ref")),
        "Quantity": qty_int,
        "UOM": clean_val(r.get("UOM")),
        "Part Number / OEM Number": clean_val(r.get("Part Number / OEM Number")),
        "Make / Brand": clean_val(r.get("Make / Brand")),
        "Specifications / Dimensions": clean_val(r.get("Specifications / Dimensions")),
    }

# Synthesize Long Text if needed
if "Long Text" not in df.columns:
    def make_long_text(r):
        parts = [f"{str(r.get('Item Description (Raw)', '')).strip()}."]
        if clean_val(r.get('Company')): parts.append(f"Company: {clean_val(r.get('Company'))}.")
        if clean_val(r.get('Item Code / Legacy Ref')): parts.append(f"Mat Code: {clean_val(r.get('Item Code / Legacy Ref'))}.")
        q, u = clean_val(r.get('Quantity')), clean_val(r.get('UOM'))
        if q and u: parts.append(f"Qty: {q} {u}.")
        elif q: parts.append(f"Qty: {q}.")
        if clean_val(r.get('Part Number / OEM Number')): parts.append(f"P/N: {clean_val(r.get('Part Number / OEM Number'))}.")
        if clean_val(r.get('Make / Brand')): parts.append(f"Make: {clean_val(r.get('Make / Brand'))}.")
        if clean_val(r.get('Specifications / Dimensions')): parts.append(f"Specs: {clean_val(r.get('Specifications / Dimensions'))}.")
        return " ".join(parts)
    df["Long Text"] = df.apply(make_long_text, axis=1)

# 3. Create Clean Train & Validation JSONL Files
shuffled = df.sample(frac=1.0, random_state=42).reset_index(drop=True)
train_df, val_df = shuffled.iloc[:4500], shuffled.iloc[4500:]

with open("train.jsonl", "w", encoding="utf-8") as f:
    for _, r in train_df.iterrows():
        item = {
            "messages": [
                {"role": "system", "content": SCHEMA_SYSTEM_PROMPT},
                {"role": "user", "content": f"Extract attributes from this industrial material text:\n\n{r['Long Text']}"},
                {"role": "assistant", "content": json.dumps(row_to_dict(r), ensure_ascii=False)},
            ]
        }
        f.write(json.dumps(item, ensure_ascii=False) + "\n")

with open("val.jsonl", "w", encoding="utf-8") as f:
    for _, r in val_df.iterrows():
        item = {
            "messages": [
                {"role": "system", "content": SCHEMA_SYSTEM_PROMPT},
                {"role": "user", "content": f"Extract attributes from this industrial material text:\n\n{r['Long Text']}"},
                {"role": "assistant", "content": json.dumps(row_to_dict(r), ensure_ascii=False)},
            ]
        }
        f.write(json.dumps(item, ensure_ascii=False) + "\n")

print(f"Created train.jsonl ({len(train_df)} rows) and val.jsonl ({len(val_df)} rows)")

# 4. Load Base Model in 4-bit NF4
BASE_MODEL_NAME = "Qwen/Qwen2.5-3B-Instruct"
OUTPUT_DIR = "./qwen2.5-3b-cpse-lora-v2"

tokenizer = AutoTokenizer.from_pretrained(BASE_MODEL_NAME, trust_remote_code=True)
if tokenizer.pad_token is None:
    tokenizer.pad_token = tokenizer.eos_token

bnb_config = BitsAndBytesConfig(
    load_in_4bit=True,
    bnb_4bit_quant_type="nf4",
    bnb_4bit_compute_dtype=torch.float16,
    bnb_4bit_use_double_quant=True,
)

model = AutoModelForCausalLM.from_pretrained(
    BASE_MODEL_NAME,
    quantization_config=bnb_config,
    device_map="auto",
    trust_remote_code=True,
)
model = prepare_model_for_kbit_training(model)

lora_config = LoraConfig(
    r=16,
    lora_alpha=32,
    target_modules=["q_proj", "k_proj", "v_proj", "o_proj", "gate_proj", "up_proj", "down_proj"],
    lora_dropout=0.05,
    bias="none",
    task_type="CAUSAL_LM",
)
model = get_peft_model(model, lora_config)

# 5. Training Configuration (Modern TRL SFTConfig with fp16=False for zero-NaN)
import functools
from trl import SFTConfig

training_args = SFTConfig(
    output_dir=OUTPUT_DIR,
    num_train_epochs=2,
    per_device_train_batch_size=2,
    per_device_eval_batch_size=2,
    gradient_accumulation_steps=4,
    learning_rate=2e-4,
    lr_scheduler_type="cosine",
    warmup_steps=30,
    logging_steps=25,
    eval_strategy="epoch",
    save_strategy="epoch",
    save_total_limit=1,
    fp16=False,          # Critical: keeps LoRA in stable float32 so no NaN occurs
    bf16=False,
    optim="paged_adamw_8bit",
    report_to="none",
    max_length=512,      # In modern TRL, max_length is defined here!
    loss_type="nll",     # Prevents chunked CE bug on k-bit models
)

# Safeguard for 4-bit partial forward wrapper
for m in [model, getattr(model, "get_base_model", lambda: model)()]:
    if hasattr(m, "forward") and isinstance(m.forward, functools.partial):
        try:
            m.forward.__func__ = m.forward.func
        except Exception:
            pass

raw_datasets = load_dataset("json", data_files={"train": "train.jsonl", "validation": "val.jsonl"})

# 6. SFTTrainer natively handles conversational 'messages' format
trainer = SFTTrainer(
    model=model,
    train_dataset=raw_datasets["train"],
    eval_dataset=raw_datasets["validation"],
    args=training_args,
)

print("\n🚀 Starting Clean QLoRA Training on CPSE Dataset...")
trainer.train()

# 7. Save & Zip New Adapter
print(f"\nSaving fine-tuned adapter to {OUTPUT_DIR}...")
trainer.model.save_pretrained(OUTPUT_DIR)
tokenizer.save_pretrained(OUTPUT_DIR)

shutil.make_archive("qwen2.5-3b-cpse-lora-v2", "zip", OUTPUT_DIR)
print("\n[✓] SUCCESS! Download 'qwen2.5-3b-cpse-lora-v2.zip' from Kaggle's Output panel!")


Loading dataset from: /kaggle/input/datasets/rakeshadak008/dataset7/CPSE_SIH26099_dataset_backup_original.xlsx
Created train.jsonl (4500 rows) and val.jsonl (506 rows)


Loading weights:   0%|          | 0/434 [00:00<?, ?it/s]

Generating train split: 0 examples [00:00, ? examples/s]

Generating validation split: 0 examples [00:00, ? examples/s]

Tokenizing train dataset:   0%|          | 0/4500 [00:00<?, ? examples/s]

...

Dropping fully masked examples from eval dataset:   0%|          | 0/506 [00:00<?, ? examples/s]

The tokenizer has new PAD/BOS/EOS tokens that differ from the model config and generation config. The model config and generation config were aligned accordingly, being updated with the tokenizer's values. Updated tokens: {'bos_token_id': None, 'pad_token_id': 151643}.



🚀 Starting Clean QLoRA Training on CPSE Dataset...


Epoch,Training Loss,Validation Loss,Entropy,Mean Token Accuracy,Num Tokens
1,0.132905,0.132467,0.135708,0.950011,1685061.000000
2,0.128966,0.129729,0.134362,0.950388,3370122.000000



Saving fine-tuned adapter to ./qwen2.5-3b-cpse-lora-v2...

[✓] SUCCESS! Download 'qwen2.5-3b-cpse-lora-v2.zip' from Kaggle's Output panel!


In [10]:
# ==============================================================================
# 🚀 ULTRA-FAST BATCHED EVALUATION FOR KAGGLE GPU T4 x2 (506 SAMPLES IN ~90 SEC)
# ==============================================================================
import json
import re
import os
import torch
import pandas as pd
from pathlib import Path
from tqdm.auto import tqdm

# 1. Ensure Model and Tokenizer are in memory
if "model" not in globals() or "tokenizer" not in globals():
    from transformers import AutoModelForCausalLM, AutoTokenizer, BitsAndBytesConfig
    from peft import PeftModel
    adapter_path = OUTPUT_DIR if "OUTPUT_DIR" in globals() else "./qwen2.5-3b-cpse-lora-v2"
    base_name = BASE_MODEL_NAME if "BASE_MODEL_NAME" in globals() else "Qwen/Qwen2.5-3B-Instruct"
    print(f"Loading base model {base_name} and LoRA adapter from {adapter_path}...")
    tokenizer = AutoTokenizer.from_pretrained(adapter_path if Path(adapter_path).exists() else base_name)
    bnb_config = BitsAndBytesConfig(load_in_4bit=True, bnb_4bit_quant_type="nf4", bnb_4bit_compute_dtype=torch.float16)
    base_model = AutoModelForCausalLM.from_pretrained(base_name, quantization_config=bnb_config, device_map="auto")
    model = PeftModel.from_pretrained(base_model, adapter_path)
    print("✓ Model and adapter successfully loaded!")

# 2. Critical for Batched Generation: Left-padding & Padding Token
tokenizer.padding_side = "left"
if tokenizer.pad_token is None:
    tokenizer.pad_token = tokenizer.eos_token

# 3. Load Unseen Validation Records
val_file = Path("/kaggle/working/val.jsonl")
if not val_file.exists():
    val_file = Path("val.jsonl")

val_records = []
with open(val_file, "r", encoding="utf-8") as f:
    for line in f:
        if line.strip():
            val_records.append(json.loads(line))

total_records = len(val_records)
print(f"Loaded {total_records} UNSEEN records from {val_file}")

# 4. Canonical Key Resolver & Evaluator Functions
KEY_MAP = {
    "Company": ["Company", "company", "Organization", "organization", "CPSE", "cpse", "Company Name", "company_name"],
    "Item Description (Raw)": ["Item Description (Raw)", "Item Description", "item_description", "Description", "description", "Item_Description", "Item", "item", "Item Name", "item_name"],
    "Item Code / Legacy Ref": ["Item Code / Legacy Ref", "Item Code", "Item_Code", "item_code", "Material Code", "material_code", "Mat Code", "mat_code", "Legacy Ref", "Code", "code"],
    "Quantity": ["Quantity", "quantity", "Qty", "qty"],
    "UOM": ["UOM", "uom", "Unit", "unit", "Unit of Measure", "Unit of Measurement"],
    "Part Number / OEM Number": ["Part Number / OEM Number", "Part Number", "part_number", "Part_Number", "Part No", "part_no", "P/N", "p/n", "OEM Number", "OEM Part Number"],
    "Make / Brand": ["Make / Brand", "Make", "make", "Brand", "brand", "Manufacturer", "manufacturer"],
    "Specifications / Dimensions": ["Specifications / Dimensions", "Specifications", "specifications", "Dimensions", "dimensions", "Specs", "specs", "Specification", "specification"],
}

def get_canonical_value(pred_dict, canonical_field):
    if not isinstance(pred_dict, dict):
        return None
    aliases = KEY_MAP.get(canonical_field, [canonical_field])
    for alias in aliases:
        if alias in pred_dict and pred_dict[alias] is not None:
            val = pred_dict[alias]
            if canonical_field == "Quantity":
                if isinstance(val, dict):
                    return val.get("Number") or val.get("number") or val.get("qty") or val.get("Quantity")
                elif isinstance(val, (str, int, float)):
                    m = re.match(r"(\d+(?:\.\d+)?)\s*(.*)", str(val).strip())
                    return m.group(1) if m else val
                return val
            if canonical_field == "UOM":
                if isinstance(val, dict):
                    return val.get("Unit") or val.get("unit") or val.get("UOM")
                return val
            return val
            
    if canonical_field == "UOM":
        for q_key in ["Quantity", "quantity", "Qty", "qty"]:
            if q_key in pred_dict and pred_dict[q_key] is not None:
                q_val = pred_dict[q_key]
                if isinstance(q_val, dict):
                    return q_val.get("Unit") or q_val.get("unit") or q_val.get("UOM")
                elif isinstance(q_val, str):
                    m = re.match(r"\d+(?:\.\d+)?\s*(.*)", q_val.strip())
                    if m and m.group(1):
                        return m.group(1).strip()
    return None

def clean_str(v):
    if v is None: return ""
    s = str(v).strip().lower().rstrip(".")
    return "" if s in ["none", "nan", "null", "unknown", "n/a"] else s

def token_f1(pred, true):
    p_tok = set(re.findall(r"\w+", clean_str(pred)))
    t_tok = set(re.findall(r"\w+", clean_str(true)))
    if not p_tok and not t_tok: return 1.0
    if not p_tok or not t_tok: return 0.0
    common = p_tok.intersection(t_tok)
    if not common: return 0.0
    prec = len(common) / len(p_tok)
    rec = len(common) / len(t_tok)
    return 2 * (prec * rec) / (prec + rec)

def eval_match(field, pred, true):
    p = clean_str(pred)
    t = clean_str(true)
    if not p and not t: return 1.0
    if field == "Quantity":
        try:
            return 1.0 if int(float(p)) == int(float(t)) else 0.0
        except Exception:
            return 1.0 if p == t else 0.0
    if field in ["Item Description (Raw)", "Specifications / Dimensions"]:
        return token_f1(p, t)
    if p and t and (p in t or t in p): return 1.0
    return 1.0 if p == t else 0.0

# 5. Fast Batched Inference Loop (Batch Size = 16)
BATCH_SIZE = 16
fields = [
    "Company",
    "Item Description (Raw)",
    "Item Code / Legacy Ref",
    "Quantity",
    "UOM",
    "Part Number / OEM Number",
    "Make / Brand",
    "Specifications / Dimensions",
]

scores = {f: [] for f in fields}
valid_json_count = 0
qualitative_samples = []

model.eval()
print(f"🚀 Running batched evaluation (Batch Size: {BATCH_SIZE}) on Dual T4 GPUs...")

with tqdm(total=total_records, desc="Evaluating Unseen Data") as pbar:
    for i in range(0, total_records, BATCH_SIZE):
        batch_records = val_records[i : i + BATCH_SIZE]
        
        # Prepare batch prompts
        prompts = [
            tokenizer.apply_chat_template(rec["messages"][:2], tokenize=False, add_generation_prompt=True)
            for rec in batch_records
        ]
        
        # Tokenize with Left-Padding
        inputs = tokenizer(prompts, return_tensors="pt", padding=True, truncation=True).to("cuda")
        
        with torch.no_grad():
            outputs = model.generate(
                **inputs,
                max_new_tokens=300,
                do_sample=False,
                pad_token_id=tokenizer.pad_token_id,
            )
            
        # Decode only the newly generated tokens
        input_len = inputs["input_ids"].shape[1]
        gen_tokens = outputs[:, input_len:]
        decoded_preds = tokenizer.batch_decode(gen_tokens, skip_special_tokens=True)
        
        for j, pred_str in enumerate(decoded_preds):
            rec = batch_records[j]
            ground_truth = json.loads(rec["messages"][2]["content"])
            raw_input_text = rec["messages"][1]["content"].replace("Extract attributes from this industrial material text:\n\n", "")
            
            # Clean JSON markdown fences
            clean_pred = pred_str.strip()
            if "```json" in clean_pred:
                clean_pred = clean_pred.split("```json")[1].split("```")[0].strip()
            elif "```" in clean_pred:
                clean_pred = clean_pred.split("```")[1].split("```")[0].strip()
                
            try:
                pred_dict = json.loads(clean_pred)
                valid_json_count += 1
            except Exception:
                pred_dict = {}
                
            pred_canonical = {f: get_canonical_value(pred_dict, f) for f in fields}
            for f in fields:
                match_score = eval_match(f, pred_canonical.get(f), ground_truth.get(f))
                scores[f].append(match_score)
                
            if len(qualitative_samples) < 3:
                qualitative_samples.append({
                    "Input": raw_input_text,
                    "Ground Truth": ground_truth,
                    "Predicted": pred_canonical,
                })
                
        pbar.update(len(batch_records))

# 6. Display Benchmark Scorecard
print("\n" + "=" * 80)
print("🎯 FULL BENCHMARK RESULTS (506 UNSEEN VALIDATION SAMPLES)")
print("=" * 80)

summary_rows = []
for f in fields:
    avg_score = (sum(scores[f]) / len(scores[f])) * 100
    metric_type = "Token F1 Score" if f in ["Item Description (Raw)", "Specifications / Dimensions"] else "Exact Match %"
    summary_rows.append({
        "Attribute": f,
        "Evaluation Metric": metric_type,
        "Accuracy Score": f"{avg_score:.2f}%",
        "Status": "⭐ Outstanding" if avg_score >= 85 else ("✅ High" if avg_score >= 70 else "⚠️ Moderate"),
    })

overall_accuracy = sum(sum(scores[f]) / len(scores[f]) for f in fields) / len(fields) * 100
summary_rows.append({
    "Attribute": "OVERALL COMPOSITE ACCURACY",
    "Evaluation Metric": "Weighted Average",
    "Accuracy Score": f"{overall_accuracy:.2f}%",
    "Status": "🏆 Production Ready" if overall_accuracy >= 85 else "✅ Solid",
})

print(f"\nJSON Syntax Validity Rate: {(valid_json_count / total_records) * 100:.2f}%\n")
display(pd.DataFrame(summary_rows))

# 7. Side-by-Side Inspection Samples
print("\n" + "=" * 80)
print("🔍 SIDE-BY-SIDE QUALITATIVE SAMPLES")
print("=" * 80)
for idx, sample in enumerate(qualitative_samples, 1):
    print(f"\n[Sample {idx}]:")
    print(f"Raw Input: {sample['Input'][:120]}...")
    cmp_df = pd.DataFrame([
        {
            "Attribute": f,
            "Ground Truth": str(sample['Ground Truth'].get(f)),
            "Model Prediction": str(sample['Predicted'].get(f)),
            "Match": "✓" if eval_match(f, sample['Predicted'].get(f), sample['Ground Truth'].get(f)) >= 0.7 else "✗"
        }
        for f in fields
    ])
    display(cmp_df)
    print("-" * 80)


Loaded 506 UNSEEN records from /kaggle/working/val.jsonl
🚀 Running batched evaluation (Batch Size: 16) on Dual T4 GPUs...


Evaluating Unseen Data:   0%|          | 0/506 [00:00<?, ?it/s]


🎯 FULL BENCHMARK RESULTS (506 UNSEEN VALIDATION SAMPLES)

JSON Syntax Validity Rate: 100.00%



,Attribute,Evaluation Metric,Accuracy Score,Status
0,Company,Exact Match %,100.00%,⭐ Outstanding
1,Item Description (Raw),Token F1 Score,100.00%,⭐ Outstanding
2,Item Code / Legacy Ref,Exact Match %,100.00%,⭐ Outstanding
3,Quantity,Exact Match %,100.00%,⭐ Outstanding
4,UOM,Exact Match %,100.00%,⭐ Outstanding
5,Part Number / OEM Number,Exact Match %,100.00%,⭐ Outstanding
6,Make / Brand,Exact Match %,100.00%,⭐ Outstanding
7,Specifications / Dimensions,Token F1 Score,100.00%,⭐ Outstanding
8,OVERALL COMPOSITE ACCURACY,Weighted Average,100.00%,🏆 Production Ready



🔍 SIDE-BY-SIDE QUALITATIVE SAMPLES

[Sample 1]:
Raw Input: bearing  ball  radial  6205  2rs. Company: Coal India (Central Coalfields Limited). Mat Code: NTPC-208406. Qty: 2590 NOS...


...

--------------------------------------------------------------------------------

[Sample 2]:
Raw Input: DIESEL EXHAUST FLUID DEF AUS 32. Company: Coal India (BCCL). Mat Code: 9231685392. Qty: 13494 LTR. P/N: CAT-731. Make: B...


,Attribute,Ground Truth,Model Prediction,Match
0,Company,Coal India (BCCL),Coal India (BCCL),✓
1,Item Description (Raw),DIESEL EXHAUST FLUID DEF AUS 32,DIESEL EXHAUST FLUID DEF AUS 32,✓
2,Item Code / Legacy Ref,9231685392,9231685392,✓
3,Quantity,13494,13494,✓
4,UOM,LTR,LTR,✓
5,Part Number / OEM Number,CAT-731,CAT-731,✓
6,Make / Brand,BOSCH,BOSCH,✓
7,Specifications / Dimensions,As per standard IS/ISO specifications. Dim: 35...,As per standard IS/ISO specifications. Dim: 35...,✓


--------------------------------------------------------------------------------

[Sample 3]:
Raw Input: industrial heater 2kw 230v tubular type any reputed make. Company: IOCL. Mat Code: ITEM-448507. Qty: 14333 NOS. Make: AN...


,Attribute,Ground Truth,Model Prediction,Match
0,Company,IOCL,IOCL,✓
1,Item Description (Raw),industrial heater 2kw 230v tubular type any re...,industrial heater 2kw 230v tubular type any re...,✓
2,Item Code / Legacy Ref,ITEM-448507,ITEM-448507,✓
3,Quantity,14333,14333,✓
4,UOM,NOS,NOS,✓
5,Part Number / OEM Number,None,None,✓
6,Make / Brand,ANY REPUTED,ANY REPUTED,✓
7,Specifications / Dimensions,As per standard IS/ISO specifications. Dim: 54...,As per standard IS/ISO specifications. Dim: 54...,✓


--------------------------------------------------------------------------------
